# Checkout Flow Without Login
This notebook demonstrates how to test the checkout flow for a guest user without logging in. It includes both dynamic testing using Cypress and static testing using Selenium.

## Dynamic Testing with Cypress
The following Cypress test ensures that a guest user can add an item to the cart and attempt to proceed to checkout.

In [ ]:
describe('Checkout Flow without Login', () => {
  it('should allow a guest user to add an item to the cart and attempt checkout', () => {
    cy.visit('http://localhost:3000/'); 

    // Select the first product and add it to the cart
    cy.get('[data-test="product-card"]').first().as('firstProduct');
    cy.get('@firstProduct').find('button[data-test="add-to-cart"]').click();

    // Check that cart count is updated
    cy.get('[data-test="cart-count"]').should('contain', '1');

    // Go to the cart page
    cy.get('[data-test="cart-link"]').click();

    // Ensure we are on the cart page
    cy.url().should('include', '/cart');

    // Ensure item is in the cart
    cy.get('[data-test="cart-item"]').should('exist');

    // Proceed to checkout
    cy.get('[data-test="checkout-button"]').click();

    // Ensure we are on the checkout page
    cy.url().should('include', '/checkout');

    // Verify guest checkout behavior
    cy.contains('Please log in or register to complete your purchase').should('be.visible');
  });
});

## Static Testing with Selenium
The following Selenium script simulates a guest user attempting to complete the checkout process by filling out the form fields and submitting the order.

In [ ]:
const { Builder, By, until } = require('selenium-webdriver');

(async function checkoutTest() {
  const driver = await new Builder().forBrowser('chrome').build();

  try {
    // Go to product page and click "Buy Now"
    await driver.get('http://localhost:3000/product/notebook-horizon-demo');
    await driver.sleep(2000); // Wait for the product page to load

    const buyNowButton = await driver.findElement(By.xpath('//button[contains(text(), "Buy Now")]'));
    await driver.wait(until.elementIsVisible(buyNowButton), 3000);
    await buyNowButton.click();
    console.log("Buy Now button clicked.");
    // Wait for redirection to checkout
    await driver.wait(until.urlContains('/checkout'), 10000);
    await driver.sleep(2000); // Wait for checkout page to stabilize

    // Fill in all form fields slowly (simulate human pace)
    const slowType = async (id, text) => {
      const field = await driver.findElement(By.id(id));
      for (const char of text) {
        await field.sendKeys(char);
        await driver.sleep(100); // Type each char slowly
      }
    };

    await slowType('name-input', 'John');
    await slowType('lastname-input', 'Doe');
    await slowType('phone-input', '1234567890');
    await slowType('email-address', 'john@example.com');
    await slowType('name-on-card', 'John Doe');
    await slowType('card-number', '4111111111111111');
    await slowType('expiration-date', '12/26');
    await slowType('cvc', '123');
    await slowType('company', 'Acme Inc.');
    await slowType('address', '123 Elm Street');
    await slowType('apartment', 'Apt 4B');
    await slowType('city', 'Springfield');
    await slowType('region', 'IL');
    await slowType('postal-code', '62704');
    await slowType('order-notice', 'Please leave the package at the door.');

    // Click "Pay Now"
    const payNowButton = await driver.findElement(By.xpath('//button[contains(text(), "Pay Now")]'));
    await driver.wait(until.elementIsVisible(payNowButton), 3000);
    await payNowButton.click();

    // Wait for confirmation or redirection
    await driver.wait(async () => {
      const url = await driver.getCurrentUrl();
      return url !== 'http://localhost:3000/checkout';
    }, 7000);

    console.log('Checkout test passed!');
  } catch (err) {
    console.error('Checkout test failed:', err.message);
  } finally {
    await driver.quit();
  }
})();